In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
import pygame
from IPython.display import Audio
from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit, ExcludeButtonsWrapper, AugmentObservation, RandomStateWrapper
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path

import logging
# import multiprocessing as mp
# mp.set_start_method("spawn", force=True)
logging.basicConfig(level=logging.DEBUG)

NUM_ENV = 1
SAVE_DIR="./model-sf4-all"

MAX_STEPS= 12_000
DETERMINISTIC=True

SAVE_DIR = Path(SAVE_DIR)

def make_env():
    def _init():
        env = make(
            "SuperStreetFighterIV-3DS",
            # render_mode="human"
        )

        # very eassy
        # env = RandomStateWrapper(env, states=[
        #     'default', 
        #     'ryu_ken_very_easy',
        # ])

        # easy
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_easy', 
        #     'ryu_ken_easy_alternate',
        # ])

        # medium 
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_medium_original', 
        #     'ryu_ken_medium_alternate',
        # ])

        # medium hard
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_medium_hard_original', 
        #     'ryu_ken_medium_hard_alternate',
        # ])

        # hard
        # env = RandomStateWrapper(env, states=[
        #     "ryu_ken_hard_original",
        #     "ryu_ken_hard_alternate",
        # ])

        # very hard
        # env = RandomStateWrapper(env, states=[
        #     "ryu_ken_very_hard_original",
        #     "ryu_ken_very_hard_alternate",
        # ])

        # hardest trainning
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_hardest_original', 
        #     'ryu_ken_hardest_alternate',
        # ])

        # all levels
        env = RandomStateWrapper(env, states=[
            'default', 
            'ryu_ken_very_easy',

            'ryu_ken_easy', 
            'ryu_ken_easy_alternate',

            'ryu_ken_medium_original', 
            'ryu_ken_medium_alternate',

            'ryu_ken_medium_hard_original', 
            'ryu_ken_medium_hard_alternate',

            "ryu_ken_hard_original",
            "ryu_ken_hard_alternate",

            "ryu_ken_very_hard_original",
            "ryu_ken_very_hard_alternate",
            
            'ryu_ken_hardest_original', 
            'ryu_ken_hardest_alternate',
        ])
        
        buttons = env.unwrapped.buttons
        to_exclude = ["START", "SELECT", "L2", "R2", "L3", "R3", "A", "X", "Y"]
        
        env = ExcludeButtonsWrapper(env, buttons, to_exclude)
        env = AugmentObservation(env)

        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=4)
        env = TimeLimit(env, max_steps=MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)
env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4, channels_order='last')

# latest_model_path = get_latest_model(SAVE_DIR)
latest_model_path = str(SAVE_DIR) + "/best_model"

print("loading from: " + str(latest_model_path))

SCREEN_WIDTH = 640*2
SCREEN_HEIGHT = 480*2
    
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

model = PPO.load(
# model = RecurrentPPO.load(
    str(latest_model_path), 
    env=env, 
    verbose=0, 
)

obs = env.reset()

while True:
    pygame.event.pump()
    action, _ = model.predict(obs, deterministic=DETERMINISTIC)

    env.render() 

    # if action[0] == 7:
    #     action[0] = 5

    obs, reward, done, info = env.step(action)

    frame = obs[0][..., -1]
    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
    frame = cv2.resize(frame, (SCREEN_WIDTH, SCREEN_HEIGHT))

    # obs = remove_bg(obs)
    
    surface = pygame.surfarray.make_surface(np.transpose(frame, (1, 0, 2)))

    window.blit(surface, (0, 0))
    pygame.display.update()

env.close()
pygame.quit()


D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
loading from: model-sf4-all/best_model


D:\Python311\Lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:78: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")
D:\Python311\Lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:259: UserWarning: You tried to call render() but no `render_mode` was passed to the env constructor.
  warnings.warn("You tried to call render() but no `render_mode` was passed to the env constructor.")
